### Следующая ячейка - блок импорта общих библиотек и формирования переменных
#### должна выполняться первой

In [ ]:
# ИМПОРТ БИБЛИОТЕК
import httpx
from pathlib import Path
import os 
from dotenv import load_dotenv
import sys
from collections import Counter

# ФОРМИРОВАНИЕ ПЕРЕМЕННЫХ
# Загружаем переменные из .env
load_dotenv()

# Получаем путь к папке с документами (данные для LLM)
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT")) 
DOCS_DIR = Path(os.getenv("OLLAMA_DOCS"))
OLLAMA_URL = os.getenv("OLLAMA_URL")
# Для импорта функций из своего файла консолидатора инструментов добавим путь по которому 
# p
sys.path.append(str(PROJECT_ROOT / "src"))

from lessons_utils import chunk_text



In [15]:
# Проверяем работоспособность сервера ollama и моделей 
# Отправляем GET-запрос и получаем ответ
response = httpx.get(f"{OLLAMA_URL}/api/tags")
# Достаём из JSON-ответа список моделей
data = response.json()
models = data.get("models", [])
print(f"Найдено моделей: {len(models)}")
for model in models:
    print(f" - {model['name']}")

Найдено моделей: 4
 - qwen2.5-coder:1.5b
 - qwen2.5-coder:7b
 - lfm3:1.7b
 - qwen2.5:1.5b


In [ ]:

# ЗАГРУЖАЕМ ДОКУМЕНТЫ
documents = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        documents.append({"path": str(path), "text": text})

print(f"Загружено документов: {len(documents)}") 

Загружено документов: 70


In [ ]:
# НАРЕЗАЕМ НА ЧАНКИ 
chunks = []
for doc in documents:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "text": chunk,
            "source": doc["path"],
        })

print(f"Всего чанков: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c['text']) for c in chunks) // len(chunks)} символов")

Всего чанков: 609
Средняя длина чанка: 540 символов


In [ ]:
# Посмотрим на несколько примеров чанков
for chunk in chunks[:3]:
    print(f"--- из {chunk['source']} ({len(chunk['text'])} символов) ---")
    print(chunk["text"][:200], "...\n")

In [22]:
# СВОДКА ПО ЧАНКАМ
chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

unique_sources = {c["source"] for c in chunks}
print(f"Файлов с чанками: {len(unique_sources)}")

source_counts = Counter(c["source"] for c in chunks)

top_3 = source_counts.most_common(3)
print("Топ-3 файла по числу чанков:")
for source, count in top_3:
    print(f"  {source}: {count} чанков") 

Всего чанков: 609
Файлов с чанками: 70
Топ-3 файла по числу чанков:
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\api.md: 74 чанков
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\faq.mdx: 38 чанков
  C:\Users\zorin_sv\ai_engineer_preproject\ollama_docs\capabilities\tool-calling.mdx: 33 чанков


In [38]:
file_path = DOCS_DIR / "api.md"
sample_text = file_path.read_text(encoding="utf-8")

for max_chars in [200, 800, 3000]:
    chunks_sample = chunk_text(sample_text, max_chars=max_chars)
    avg_length = sum(len(c) for c in chunks_sample) // len(chunks_sample) if chunks_sample else 0
    print(f"max_chars = {max_chars} \n -> чанков: {len(chunks_sample)}, средняя длина: {avg_length} ")


max_chars = 200 
 -> чанков: 161, средняя длина: 330 
max_chars = 800 
 -> чанков: 74, средняя длина: 736 
max_chars = 3000 
 -> чанков: 30, средняя длина: 1818 
